# Visualization 1 — The Spaghetti Line Chart

**Goal:** Show the evolution of French baby name popularity from 1900 to 2020.  
All names start as faint gray lines; a **regex search box** lets you highlight specific names in distinct bold colors.

**Design (from 3_Visualizations.pdf)**
- X-axis: Year (1900–2020)
- Y-axis: Total births per year (summed across all departments and both genders)
- Background: all names as thin gray lines (opacity ≈ 15 %)
- Highlight: matching names drawn thick and colored, with end-of-line labels
- Interaction: text input accepts a name or a pipe-separated regex (`LEO|ALICE|KEVIN`)

**Data:** [dpt2020.csv](https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv)  
Columns: `sexe` (1=M, 2=F) · `preusuel` (name) · `annais` (year) · `dpt` (department code) · `nombre` (count)

In [ ]:
import pandas as pd
import altair as alt

print(f"pandas  {pd.__version__}")
print(f"altair  {alt.__version__}")

## 1 · Load & preprocess

In [ ]:
URL = "https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv"

df = pd.read_csv(URL, sep=';', dtype={'annais': str, 'dpt': str})
print(f"Raw rows: {len(df):,}")
df.head()

In [ ]:
# Drop the aggregate 'rare names' catch-all bucket
df = df[df['preusuel'] != '_PRENOMS_RARES']

# Drop rows with unknown year ('XXXX') and cast to int
df = df[df['annais'] != 'XXXX'].copy()
df['annais'] = df['annais'].astype(int)
df = df[(df['annais'] >= 1900) & (df['annais'] <= 2020)]

# Aggregate: total births per (name, year) — sums across all departments and genders
yearly = (
    df.groupby(['preusuel', 'annais'])['nombre']
    .sum()
    .reset_index()
    .rename(columns={'preusuel': 'name', 'annais': 'year', 'nombre': 'count'})
)

print(f"Unique names : {yearly['name'].nunique():,}")
print(f"Year range   : {yearly['year'].min()} – {yearly['year'].max()}")
print(f"Total rows   : {len(yearly):,}")

In [ ]:
# Keep only the top-N names by cumulative births to keep the chart responsive.
TOP_N = 300

top_names = (
    yearly.groupby('name')['count']
    .sum()
    .nlargest(TOP_N)
    .index.tolist()
)

# Exclude count <= 3: INSEE records rare occurrences as exactly 3 (privacy floor),
# which causes hundreds of names to pile up on the same Y position and form a
# visible horizontal strip at the bottom of the spaghetti chart.
df_plot = yearly[(yearly['name'].isin(top_names)) & (yearly['count'] > 3)].copy()
print(f"Plotting {TOP_N} names × {yearly['year'].nunique()} years = {len(df_plot):,} rows")

## 2 · Build the chart

In [ ]:
# Lift Altair's default 5 000-row cap
alt.data_transformers.disable_max_rows()

# ── Interactive parameter: text input for name search ────────────────────────
search = alt.param(
    name='highlight',
    value='LEO|ALICE|MAEL',
    bind=alt.binding(
        input='text',
        placeholder='e.g.  LEO|ALICE|KEVIN',
        name='Highlight names (| separated):  '
    )
)

# Exact-match filter — "LEO" matches only LEO, not LEON/LEONIE/…
REGEX_FILTER = "test(regexp('^(' + replace(highlight, ' ', '') + ')$', 'i'), datum.name)"

# ── Two separate sources ──────────────────────────────────────────────────────
# Both layers use the same count > 3 floor so no data point falls below domainMin=4.
yearly_fg = yearly[yearly['count'] > 3]
base_bg = alt.Chart(df_plot)    # top-300 gray backdrop  (already filtered count > 3)
base_fg = alt.Chart(yearly_fg)  # all names, highlighted (filtered count > 3)

x_enc = alt.X('year:Q', title='Year', axis=alt.Axis(format='d', tickCount=13))

# domainMin=4 matches the count > 3 filter applied to both layers, so the axis
# floor aligns with the lowest data point and no line can overflow below the X-axis.
y_enc = alt.Y(
    'count:Q',
    title='Total births (log scale)',
    scale=alt.Scale(type='log', zero=False, domainMin=4, nice=False),
    axis=alt.Axis(format='~s')
)

TOOLTIP = [
    alt.Tooltip('name:N',  title='Name'),
    alt.Tooltip('year:Q',  title='Year',         format='d'),
    alt.Tooltip('count:Q', title='Total births', format=',')
]

# ── Layer 1 · ALL top-300 names — faint gray spaghetti ───────────────────────
bg = base_bg.mark_line(
    strokeWidth=0.8,
    opacity=0.15,
    color='#888888'
).encode(
    x=x_enc,
    y=y_enc,
    detail='name:N',
    tooltip=TOOLTIP
)

# ── Layer 2 · MATCHING names — colored, thick lines ──────────────────────────
fg = (
    base_fg.mark_line(strokeWidth=2.8, opacity=0.9)
    .encode(
        x=x_enc,
        y=y_enc,
        color=alt.Color(
            'name:N',
            scale=alt.Scale(scheme='tableau20'),
            legend=alt.Legend(title='Highlighted names', orient='top-right')
        ),
        detail='name:N',
        tooltip=TOOLTIP
    )
    .transform_filter(REGEX_FILTER)
)

# ── Layer 3 · End-of-line labels ─────────────────────────────────────────────
lbl = (
    base_fg.mark_text(align='left', dx=6, dy=-3, fontSize=11, fontWeight='bold')
    .encode(
        x=x_enc,
        y=y_enc,
        color=alt.Color('name:N', scale=alt.Scale(scheme='tableau20'), legend=None),
        text='name:N'
    )
    .transform_filter(REGEX_FILTER)
    .transform_joinaggregate(max_year='max(year)', groupby=['name'])
    .transform_filter('datum.year == datum.max_year')
)

# ── Compose ───────────────────────────────────────────────────────────────────
chart = (
    (bg + fg + lbl)
    .add_params(search)
    .properties(
        title=alt.TitleParams(
            text='French Baby Names — Evolution Over Time (1900–2020)',
            subtitle='Type exact names separated by | (e.g. LEO|ALICE|KEVIN) — Y axis is logarithmic',
            anchor='start',
            fontSize=16,
            fontWeight='bold',
            subtitleFontSize=12,
            subtitleColor='#666666'
        ),
        width=760,
        height=450,
        padding={'left': 10, 'right': 90, 'top': 10, 'bottom': 10}
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridOpacity=0.2, gridColor='#e0e0e0')
)

chart

## 3 · Save to HTML (optional)

In [ ]:
chart.save('visualization_1_spaghetti.html')
print('Saved to visualization_1_spaghetti.html')